# Lesson LIN 2: Matrices as Spatial Operators (Montages, Rereferencing, & Rank) [SOLUTIONS GUIDE]
**Foundations · Applied Linear Algebra for Neural Arrays**  
*Pedagogical Structure: 3Blue1Brown Linear Transformations + Multimodal Electrophysiology*

---

> **INSTRUCTOR / SOLUTIONS GUIDE**: Full verified implementations and mathematical proofs for Lesson LIN 2.

## 1. The 3B1B View: Matrices as Coordinate Transformations
In 3Blue1Brown, matrix multiplication $y = M x$ is not a set of row-times-column rules to memorize. A matrix is a **Linear Transformation of Space**:
- The columns of $M$ tell you where the original basis vectors land after transformation.
- In multi-channel electrophysiology, your raw data matrix $X_{\text{raw}}$ has dimensions $[C_{\text{raw}} \times N_{\text{samples}}]$.
- Applying a montage (re-referencing) is multiplying by a derivation matrix $M$ of size $[C_{\text{deriv}} \times C_{\text{raw}}]$:
  $$X_{\text{reref}} = M X_{\text{raw}}$$
  $$\begin{bmatrix} C_{\text{deriv}} \times N \end{bmatrix} = \begin{bmatrix} C_{\text{deriv}} \times C_{\text{raw}} \end{bmatrix} \begin{bmatrix} C_{\text{raw}} \times N \end{bmatrix}$$

Each row of $M$ defines a single output derivation as a weighted spatial combination of recorded channels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.edgecolor'] = '#404040'
plt.rcParams['axes.linewidth'] = 0.8

print('Environment initialized for Lesson LIN 2!')

---
### STEP 1: Directional DBS (1-3-3-1 Geometry)
Modern DBS leads (Medtronic SenSight, Boston Scientific Cartesia, Abbott Directed) have **8 contacts** arranged in 4 levels:
- **Level 1 (Ventral)**: Omnidirectional Ring 1
- **Level 2**: 3 directional segments at $120^\circ$ ($2A, 2B, 2C$)
- **Level 3**: 3 directional segments at $120^\circ$ ($3A, 3B, 3C$)
- **Level 4 (Dorsal)**: Omnidirectional Ring 4

```
   Contact Index:   0    1    2    3    4    5    6    7
   Contact Name:  [ 1,  2A,  2B,  2C,  3A,  3B,  3C,   4 ]
```

#### The Vertical Bipolar Montage
To reject shared common-mode reference noise while preserving directional selectivity along the shank, we reference each segmented contact against its nearest ring:
- Level 2 segments against Ring 1: $(2A - 1), (2B - 1), (2C - 1)$
- Level 3 segments against Ring 4: $(3A - 4), (3B - 4), (3C - 4)$
- Plus the vertical span: $(4 - 1)$

This is the `bipolar_vertical` scheme in `src/dbsspeech/preprocess/reference.py`, built there by `build_montage` from the lead geometry declared in `configs/leads.yaml`. MNE's equivalent is `mne.set_bipolar_reference`.

**What this montage costs, which is not nothing.** It cancels the amplifier's shared reference exactly, but it does not leave you with independent derivations. Rows 0 to 2 all contain $-1 \times$ Ring 1, so Ring 1 has become a new shared term across three "directional" channels. Feed *fully independent* unit-variance channels through $M_{\text{vertical}}$ and the resulting derivations come out correlated at $r \approx 0.5$ within each group of three. Any noise or any source local to Ring 1 now appears on all three level-2 derivations at once, which looks exactly like a spatially broad effect.

So the choice is a tradeoff, not an upgrade, and the package states it:

| Scheme | Common-mode rejection | Cost |
|---|---|---|
| `bipolar_vertical` | complete | shares a ring across three derivations |
| `bipolar_horizontal` ($a-b$, $b-c$, $c-a$) | strongest | destroys per-segment values, so no directional claim can be made from it |
| `car` | complete | averaging over few contacts spreads a strong local signal into every derivation |

`bipolar_vertical` is used here because it preserves per-segment identity, which any directional analysis needs. That is the reason to accept the induced correlation, not a reason to ignore it.

**Task 1**: Construct the $[7 \times 8]$ vertical bipolar transformation matrix $M_{\text{vertical}}$ from scratch.

In [ ]:
def build_vertical_bipolar_matrix() -> np.ndarray:
    """Construct the 7x8 vertical bipolar montage matrix for a 1-3-3-1 directional DBS lead."""
    M = np.zeros((7, 8), dtype=float)
    # Level 2 segments minus Ring 1
    M[0, 1] = 1.0; M[0, 0] = -1.0  # 2A - 1
    M[1, 2] = 1.0; M[1, 0] = -1.0  # 2B - 1
    M[2, 3] = 1.0; M[2, 0] = -1.0  # 2C - 1
    # Level 3 segments minus Ring 4
    M[3, 4] = 1.0; M[3, 7] = -1.0  # 3A - 4
    M[4, 5] = 1.0; M[4, 7] = -1.0  # 3B - 4
    M[5, 6] = 1.0; M[5, 7] = -1.0  # 3C - 4
    # Ring 4 minus Ring 1
    M[6, 7] = 1.0; M[6, 0] = -1.0  # 4 - 1
    return M


In [ ]:
# --- TEST CELL FOR STEP 1 ---
M_vert = build_vertical_bipolar_matrix()
assert M_vert.shape == (7, 8), f'Expected shape (7, 8), got {M_vert.shape}'
# Verify each row sums to zero (mandatory for any bipolar derivation to reject DC common mode)
row_sums = np.sum(M_vert, axis=1)
assert np.allclose(row_sums, 0.0), f'Every bipolar derivation row must sum to zero: got {row_sums}'
# Check every derivation element-wise. Row sums and two spot checks are not
# enough: a flipped row still sums to zero, and a duplicated row does too, so
# both pass a sum test while the montage is wrong.
DERIVATIONS = [(1, 0), (2, 0), (3, 0), (4, 7), (5, 7), (6, 7), (7, 0)]  # (plus, minus)
expected = np.zeros((7, 8))
for row, (plus, minus) in enumerate(DERIVATIONS):
    expected[row, plus] = 1.0
    expected[row, minus] = -1.0

assert np.array_equal(M_vert, expected), (
    'every derivation must be exactly right, not merely zero-summing.\n'
    f'got:\n{M_vert}\nexpected:\n{expected}'
)
# And no derivation may be repeated, which would silently drop a contact.
assert len({tuple(r) for r in M_vert}) == 7, 'two rows are identical: a derivation is missing'
print('✅ Step 1 Passed! Directional DBS vertical bipolar matrix correctly constructed.')

---
### STEP 2: Common Average Referencing (CAR) in Matrix Form
On subdural **ECoG grids** (e.g. 32 or 64 contacts), a standard montage is Common Average Referencing (CAR). For each channel $i$, we subtract the instantaneous mean across all $C$ channels:
$$V_{\text{CAR}}[i] = V[i] - \frac{1}{C} \sum_{j=1}^C V[j]$$

In linear algebra, this is an **Orthogonal Projection Matrix**:
$$M_{\text{CAR}} = I_C - \frac{1}{C} \mathbf{1} \mathbf{1}^T$$
Where $I_C$ is the $[C \times C]$ identity matrix and $\mathbf{1}$ is a column vector of ones $[C \times 1]$.

**Task 2**: Implement `build_car_matrix(n_channels)` using `np.eye` and `np.ones`.

In [ ]:
def build_car_matrix(n_channels: int) -> np.ndarray:
    """Construct the [C x C] Common Average Referencing projection matrix."""
    I = np.eye(n_channels)
    ones_matrix = np.ones((n_channels, n_channels)) / n_channels
    return I - ones_matrix


In [ ]:
# --- TEST CELL FOR STEP 2 ---
C = 8
M_car = build_car_matrix(C)
assert M_car.shape == (C, C), f'Expected shape ({C}, {C}), got {M_car.shape}'
# Test idempotent projection property: M_car @ M_car == M_car (projecting twice does nothing new)
assert np.allclose(M_car @ M_car, M_car), 'CAR matrix must be an idempotent projection matrix!'
# Test row sums to zero
assert np.allclose(np.sum(M_car, axis=1), 0.0), 'Each row of CAR must sum to zero'
print('✅ Step 2 Passed! CAR projection matrix satisfies all algebraic properties.')

---
### STEP 3: The Rank Deficiency Proof
**What is Matrix Rank?**
Rank is the true number of independent spatial dimensions in your recording. A full-rank $[C \times C]$ matrix has rank $C$.

**The Mathematical Theorem**:
Because CAR subtracts the mean across all channels, the sum of all re-referenced channels is identically zero:
$$\sum_{i=1}^C V_{\text{CAR}}[i] = 0 \implies \text{Row } C = -\sum_{i=1}^{C-1} \text{Row } i$$
Therefore, one channel is completely redundant. **CAR reduces matrix rank from $C$ to $C - 1$**.

A rank-deficient matrix is singular, so its inverse does not exist. It is natural to assume NumPy will say so. It will not.

`np.linalg.inv` factorizes numerically, and floating-point round-off means the pivot it would have to reject is essentially never exactly zero. Called on a CAR covariance matrix it typically returns **without raising anything at all**, handing back an array of amplified round-off. The determinant comes back as something like $-2 \times 10^{-16}$ rather than $0$, and the condition number runs to $10^{15}$.

The failure is silent rather than loud, and that is the dangerous part: a wrong answer that looks like an answer. Check `np.linalg.matrix_rank` or the condition number *before* you invert, instead of waiting for an exception that never arrives. Lesson LIN 3 measures exactly this.

**Task 3**: Write a test verifying that `np.linalg.matrix_rank` of $M_{\text{CAR}}$ equals $C - 1$.

In [ ]:
# --- VERIFYING RANK DEFICIENCY IN CODE ---
for n_ch in [4, 8, 16, 64]:
    M = build_car_matrix(n_ch)
    rank = np.linalg.matrix_rank(M)
    assert rank == n_ch - 1, f'For {n_ch} channels, rank must be {n_ch - 1}, but got {rank}'

print('✅ Step 3 Passed! Proved that CAR mathematically reduces data rank by exactly 1.')

---
## 4. Electrophysiology Simulation: Eliminating Common-Mode Reference Noise
Let's test our matrices on simulated 8-contact directional DBS data:
1. **Local Neural Activity**: A true local 20 Hz beta oscillation localized to contact 2A (dorsolateral STN).
2. **Shared Reference Noise**: A huge $100\ \mu\text{V}$ 60 Hz line-noise artifact and slow movement drift present equally on ALL 8 contacts.

We will apply the vertical bipolar transformation $X_{\text{bipolar}} = M_{\text{vertical}} X_{\text{raw}}$ and observe the common-mode noise cancellation.

In [ ]:
np.random.seed(42)
srate = 1000
time = np.linspace(0, 0.5, int(srate * 0.5))
n_samples = len(time)

# 1. Shared reference artifact (60 Hz + drift) present across all channels
common_mode_artifact = 80.0 * np.sin(2 * np.pi * 60 * time) + 20.0 * np.sin(2 * np.pi * 3 * time)

# 2. Raw 8-channel matrix [8 x N]
X_raw = np.zeros((8, n_samples))
for ch in range(8):
    X_raw[ch, :] = common_mode_artifact + np.random.normal(0, 3.0, n_samples)

# 3. Plant true local 20 Hz beta signal strictly on Contact 2A (index 1)
local_beta = 35.0 * np.sin(2 * np.pi * 20 * time)
X_raw[1, :] += local_beta

# 4. Apply Vertical Bipolar Matrix Transformation: X_bipolar = M @ X_raw
M_vert = build_vertical_bipolar_matrix()
X_bipolar = M_vert @ X_raw

# Plotting: Raw Monopolar vs Bipolar Derivations
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Raw Monopolar Channels
for ch in range(8):
    ax1.plot(time * 1000, X_raw[ch, :] + ch * 60, color='#737373', lw=1.0)
ax1.set_ylabel('Channels (Offset μV)')
ax1.set_title('Raw Monopolar Channels: Completely Swamped by 60 Hz Shared Reference Noise (Guardrail G1)', fontweight='bold')
ax1.grid(True, alpha=0.3, ls='--')

# Vertical Bipolar Channels
deriv_names = ['2A - 1', '2B - 1', '2C - 1', '3A - 4', '3B - 4', '3C - 4', '4 - 1']
for d in range(7):
    color = '#06b6d4' if d == 0 else '#a855f7'
    lw = 2.0 if d == 0 else 1.0
    ax2.plot(time * 1000, X_bipolar[d, :] + d * 40, color=color, lw=lw, label=deriv_names[d] if d == 0 else '')
ax2.set_ylabel('Derivations (Offset μV)')
ax2.set_xlabel('Time (ms)')
ax2.set_title('Vertical Bipolar Transformation (M @ X): Common-Mode Cleaned; True 20 Hz Beta Emerges on 2A-1', fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3, ls='--')

plt.tight_layout()
plt.show()

---
## Summary of What You Mastered in Lesson LIN 2
1. **Matrices as Montages**: Re-referencing is a matrix transformation $X_{\text{reref}} = M X_{\text{raw}}$.
2. **Directional DBS Geometry**: Vertical bipolar derivations reference directional segments against the nearest ring, preserving orientation while rejecting common-mode noise.
3. **CAR and Idempotent Projections**: $M_{\text{CAR}} = I - \frac{1}{C}\mathbf{1}\mathbf{1}^T$ projects signals into a zero-sum subspace.
4. **Rank Deficiency**: CAR mathematically reduces spatial degrees of freedom by 1 ($C \to C-1$).

Next up: **Lesson LIN 3 — Matrix Inverses, Conditioning, and Multicollinearity in Source Localization**!